# 使用scGPT进行扰动数据分析

本教程演示如何使用scGPT模型进行扰动数据分析。扰动数据通常包括CRISPR筛选、药物处理或基因敲低等实验数据，分析这些数据可以帮助我们理解基因功能和细胞对扰动的响应。


In [1]:
# 导入必要的库
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# 添加scGPT路径
sys.path.insert(0, "../")
import scgpt as scg

print("✅ 所有库加载完成！")

In [2]:
# 设置参数
model_dir = Path("../save/scGPT_human")
data_dir = Path("../data/perturbation")
save_dir = Path("../results/perturbation")

# 创建保存目录
save_dir.mkdir(parents=True, exist_ok=True)

print("✅ 参数设置完成！")

In [3]:
# 加载扰动数据
adata = sc.read_h5ad(data_dir / "perturbation_data.h5ad")

# 查看数据信息
print(f"数据形状: {adata.shape}")
print(f"扰动类型: {adata.obs['perturbation'].unique()}")
print(f"细胞类型: {adata.obs['cell_type'].unique()}")

# 预处理
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print("✅ 扰动数据加载完成！")

In [4]:
# 使用scGPT分析扰动响应
perturbation_results = scg.tasks.analyze_perturbation(
    adata,
    model_dir,
    perturbation_key="perturbation",
    control_group="control",
    batch_size=32,
)

print("✅ 扰动分析完成！")

In [5]:
# 分析差异表达基因
de_genes = perturbation_results["differential_genes"]

print("差异表达基因分析:")
for perturbation, genes in de_genes.items():
    print(f"\n{perturbation}:")
    print(f"  上调基因: {len(genes['up'])}")
    print(f"  下调基因: {len(genes['down'])}")
    print(f"  Top上调基因: {genes['up'][:5]}")
    print(f"  Top下调基因: {genes['down'][:5]}")

# 可视化扰动效果
plt.figure(figsize=(12, 8))
sc.pp.neighbors(adata, use_rep="X_scgpt")
sc.tl.umap(adata)
sc.pl.umap(adata, color=["perturbation", "cell_type"], ncols=2, frameon=False)
plt.suptitle("扰动数据可视化")
plt.tight_layout()
plt.show()

print("✅ 扰动分析完成！")

In [6]:
# 保存结果
# 保存差异表达基因
for perturbation, genes in de_genes.items():
    pd.DataFrame({"up_genes": genes["up"], "down_genes": genes["down"]}).to_csv(
        save_dir / f"{perturbation}_de_genes.csv", index=False
    )

# 保存分析后的adata
adata.write_h5ad(save_dir / "perturbation_analysis.h5ad")

print("✅ 扰动分析结果已保存！")